In [ ]:
#!pip install -q langchain-openai langchain-core requests -q

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage , ToolMessage
import requests
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = 'xxxxxxxxxxxxxxxxxxxxxx'

In [ ]:
# --------------------------
# Define arithmetic tools
# --------------------------

@tool
def add(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their sum"""
    return a + b

In [ ]:
@tool
def subtract(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their difference (a - b)"""
    return a - b

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [ ]:
@tool
def divide(a: int, b: int) -> float:
    """Given 2 numbers a and b this tool returns their quotient (a / b). Raises error if b = 0."""
    if b == 0:
        raise ValueError("Division by zero is not allowed")
    return a / b

In [ ]:
# --------------------------
# Test tools directly
# --------------------------
print("Add:", add.invoke({'a': 10, 'b': 5}))
print("Subtract:", subtract.invoke({'a': 10, 'b': 5}))
print("Multiply:", multiply.invoke({'a': 10, 'b': 5}))
print("Divide:", divide.invoke({'a': 10, 'b': 5}))

In [ ]:
# --------------------------
# Bind tools to LLM
# --------------------------

llm = ChatOpenAI(model='gpt-4o-mini')

# Attach all tools
llm_with_tools = llm.bind_tools([add, subtract, multiply, divide])

In [ ]:
# --------------------------
# Example usage with LLM
# --------------------------

query = "Can you divide 100 by 25?"
messages = [
              HumanMessage(query)
          ]

In [ ]:
# Let LLM decide which tool to call
result = llm_with_tools.invoke(messages)
messages.append(result)

In [ ]:
messages

In [ ]:
# Step 3: Actually run the tool
tool_result = multiply.invoke(result.tool_calls[0])

In [ ]:
# Step 4: Wrap tool output in a ToolMessage with the same tool_call_id
messages.append(
    ToolMessage(
        content=str(tool_result),                 # tool output
        tool_call_id=result.tool_calls[0]["id"]   # must match LLM's tool_call_id
    )
)

In [ ]:
# Step 5: Now ask LLM for final answer
final_answer = llm_with_tools.invoke(messages)
print("\nFinal Answer from LLM:", final_answer.content)

# **Agentic Approach**

In [ ]:

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import initialize_agent, AgentType

In [ ]:
# --------------------------
# Define arithmetic tools
# --------------------------

@tool
def add(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their sum"""
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their difference (a - b)"""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Given 2 numbers a and b this tool returns their quotient (a / b). Raises error if b = 0."""
    if b == 0:
        raise ValueError("Division by zero is not allowed")
    return a / b

In [ ]:
# --------------------------
# Setup LLM + Agent
# --------------------------

llm = ChatOpenAI(model="gpt-4o-mini")
tools = [add, subtract, multiply, divide]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=False, # true
    handle_parsing_errors=True
)

In [ ]:
#  query
query1 = "What is 5 plus 6? explain with reason"
response1 = agent.run(query1)
print("\nAgent Response 1:", response1)


# **Manual tool execution +  Agent-based execution**

In [ ]:
from langchain_core.tools import tool, InjectedToolArg
from typing import Annotated
import requests

- create API key : https://www.exchangerate-api.com/
- API key : 
- Request :


In [ ]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Fetch the currency conversion factor between a base currency and a target currency.
    """
    url = f'https://v6.exchangerate-api.com/v6/xxxxxxxxxxxxxxxx/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

In [ ]:
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Convert base currency amount into target currency using the given conversion rate.
    """
    return base_currency_value * conversion_rate

Bind Tools to LLM (manual tool call mode)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
import json

llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

Step 3: Multi-step Query (Manual Execution)

In [ ]:
messages = [HumanMessage("What is the conversion factor between INR and USD, and based on that can you convert 10 INR to USD?")]

# Step 1: Ask LLM
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

conversion_rate = None


In [ ]:

# Step 2: Execute tool calls
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        # Run API tool
        print('Excuted #1')
        tool_response = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_response.content)['conversion_rate']
        messages.append(
            ToolMessage(content=tool_response.content, tool_call_id=tool_call["id"])
        )
    elif tool_call['name'] == 'convert':
        # Insert conversion_rate dynamically
        print('Excuted #2')
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_response = convert.invoke(tool_call)
        messages.append(
            ToolMessage(content=str(tool_response), tool_call_id=tool_call["id"])
        )

# Step 3: Final Answer from LLM
final_answer = llm_with_tools.invoke(messages)
print("Final Answer:", final_answer.content)
